# Task 4: Compare LSTM with the other models
We have three models now:
Naive Bayes, Logistic Regression and LSTM.
All of them were tested on the same 1,026 messages.
Here we compare them and look at the messages they got wrong.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix
nb = pd.read_csv('../data/pred_nb.csv')
lr = pd.read_csv('../data/pred_lr.csv')
lstm = pd.read_csv('../data/pred_lstm.csv')
print(nb.shape, lr.shape, lstm.shape)
print('Same messages:', (nb['text'] == lr['text']).all(),
      (nb['text'] == lstm['text']).all())
y = (nb['label'] == 'spam').astype(int)
print('Spam in test:', y.sum())

(1026, 4) (1026, 4) (1026, 4)
Same messages: True True
Spam in test: 125


## 1. The three models side by side
We also add a very simple model that always says "ham",
to see what our models really add.

In [2]:
models = [('Naive Bayes', nb), ('Logistic Regression', lr), ('LSTM', lstm)]
rows = []
for name, d in models:
    p = (d['pred'] == 'spam').astype(int)
    cm = confusion_matrix(y, p)
    rows.append([name,
                 round(100 * accuracy_score(y, p), 1),
                 round(100 * precision_score(y, p), 1),
                 round(100 * recall_score(y, p), 1),
                 round(100 * f1_score(y, p), 1),
                 cm[0, 1], cm[1, 0]])
rows.append(['Always say ham', round(100 * (y == 0).mean(), 1),
             0.0, 0.0, 0.0, 0, int(y.sum())])
table = pd.DataFrame(rows, columns=['model', 'accuracy', 'precision',
                                    'recall', 'f1', 'false_alarms', 'missed'])
print(table.to_string(index=False))

              model  accuracy  precision  recall   f1  false_alarms  missed
        Naive Bayes      98.9      100.0    91.2 95.4             0      11
Logistic Regression      99.1       97.5    95.2 96.4             3       6
               LSTM      98.9       95.2    96.0 95.6             6       5
     Always say ham      87.8        0.0     0.0  0.0             0     125


## 2. Do the models agree?
We put the three answers next to each other,
and we count how many models said "spam" for each message.

In [3]:
both = nb[['label', 'text']].copy()
both['nb'] = nb['pred']
both['lr'] = lr['pred']
both['lstm'] = lstm['pred']
both['votes'] = (both[['nb', 'lr', 'lstm']] == 'spam').sum(axis=1)
agree = (both['votes'] == 0) | (both['votes'] == 3)
print('All three agree on:', agree.sum(), 'messages')
print('They disagree on :', (~agree).sum(), 'messages')
print()
print(both[~agree].groupby(['label', 'nb', 'lr', 'lstm']).size())

All three agree on: 1010 messages
They disagree on : 16 messages

label  nb    lr    lstm
ham    ham   ham   spam    5
             spam  ham     2
                   spam    1
spam   ham   ham   spam    2
             spam  spam    5
       spam  spam  ham     1
dtype: int64


## 3. The messages where they disagree
We look at the real messages, to understand why.

In [4]:
for label in ['spam', 'ham']:
    print('=' * 15, 'TRUE', label.upper())
    sub = both[(~agree) & (both['label'] == label)]
    for r in sub.itertuples():
        print('nb:', r.nb, '| lr:', r.lr, '| lstm:', r.lstm)
        print('   ', r.text[:110])
    print()

=============== TRUE SPAM
nb: ham | lr: ham | lstm: spam
    TBS/PERSOLVO. been chasing us since Sept for£38 definitely not paying now thanks to your information. We will 
nb: spam | lr: spam | lstm: ham
    RECPT 1/3. You have ordered a Ringtone. Your order is being processed...
nb: ham | lr: spam | lstm: spam
    SMS. ac JSco: Energy is high, but u may not know where 2channel it. 2day ur leadership skills r strong. Psychi
nb: ham | lr: spam | lstm: spam
    You can donate £2.50 to UNICEF's Asian Tsunami disaster support fund by texting DONATE to 864233. £2.50 will b
nb: ham | lr: ham | lstm: spam
    Burger King - Wanna play footy at a top stadium? Get 2 Burger King before 1st Sept and go Large or Super with 
nb: ham | lr: spam | lstm: spam
    Ur balance is now £600. Next question: Complete the landmark, Big, A. Bob, B. Barry or C. Ben ?. Text A, B or 
nb: ham | lr: spam | lstm: spam
    Hi I'm sue. I am 20 years old and work as a lapdancer. I love sex. Text me live - I'm i my bedro

## 4. Why does the LSTM give false alarms?
We think the false alarms are messages with strange or rare words.
We check this: we count the words that are not in the train data.

In [5]:
train = pd.read_csv('../data/train_prep_v2.csv')
test = pd.read_csv('../data/test_prep_v2.csv')
known = set(' '.join(train['clean_text']).split())

def new_word_share(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return 100 * sum(w not in known for w in words) / len(words)

both['new_words'] = test['clean_text'].apply(new_word_share)
lstm_false = (both['label'] == 'ham') & (both['lstm'] == 'spam')
print('All ham messages      :', round(both[both['label'] == 'ham']['new_words'].mean(), 1), '% new words')
print('Ham that LSTM got wrong:', round(both[lstm_false]['new_words'].mean(), 1), '% new words')
print('Number of these messages:', lstm_false.sum())

All ham messages      : 6.3 % new words
Ham that LSTM got wrong: 25.0 % new words
Number of these messages: 6


## 5. Save the comparison
Abdullah will use this table in the report and in the charts.

In [6]:
table.to_csv('../data/model_comparison.csv', index=False)
both.to_csv('../data/all_predictions.csv', index=False)
print('Saved')
print(table.to_string(index=False))

Saved
              model  accuracy  precision  recall   f1  false_alarms  missed
        Naive Bayes      98.9      100.0    91.2 95.4             0      11
Logistic Regression      99.1       97.5    95.2 96.4             3       6
               LSTM      98.9       95.2    96.0 95.6             6       5
     Always say ham      87.8        0.0     0.0  0.0             0     125
